# Goldilocks pilot 03 — shadow regulator
Rule-based regulator with hysteresis. It recommends interventions but never controls execution authority.

In [ ]:
from dataclasses import dataclass, asdict
from enum import Enum
import json, time
import numpy as np
import pandas as pd


In [ ]:
class Action(str, Enum):
    CONTINUE = 'continue'
    NOVELTY_PULSE = 'novelty_pulse'
    REANCHOR = 'reanchor'
    HALT_AND_RESET = 'halt_and_reset'

@dataclass
class Thresholds:
    d_low_enter: float = 0.04
    d_low_exit: float = 0.06
    c_low_enter: float = 0.70
    c_low_exit: float = 0.78
    r_low_enter: float = 0.82
    r_low_exit: float = 0.88
    v_high_enter: float = 0.12
    v_high_exit: float = 0.09

@dataclass
class State:
    diversity: float
    coherence: float
    returnability: float
    velocity: float
    purpose_integrity: float
    epistemic_gain: float = 0.0

class ShadowRegulator:
    def __init__(self, thresholds=Thresholds()):
        self.t = thresholds
        self.active = Action.CONTINUE

    def decide(self, s: State):
        previous = self.active
        if s.purpose_integrity < 1.0 or s.velocity > self.t.v_high_enter:
            self.active = Action.HALT_AND_RESET
        elif s.coherence < self.t.c_low_enter or s.returnability < self.t.r_low_enter:
            self.active = Action.REANCHOR
        elif s.diversity < self.t.d_low_enter and s.coherence >= self.t.c_low_exit:
            self.active = Action.NOVELTY_PULSE
        elif self.active == Action.HALT_AND_RESET and s.velocity <= self.t.v_high_exit and s.purpose_integrity == 1.0:
            self.active = Action.REANCHOR
        elif self.active == Action.REANCHOR and s.coherence >= self.t.c_low_exit and s.returnability >= self.t.r_low_exit:
            self.active = Action.CONTINUE
        elif self.active == Action.NOVELTY_PULSE and s.diversity >= self.t.d_low_exit:
            self.active = Action.CONTINUE
        return {
            'timestamp_ns': time.time_ns(),
            'previous_action': previous.value,
            'recommended_action': self.active.value,
            'state': asdict(s),
            'execution_authority': False,
        }


In [ ]:
trajectory = [
    State(0.02, 0.90, 0.95, 0.03, 1.0),  # stasis
    State(0.07, 0.86, 0.92, 0.06, 1.0),  # productive movement
    State(0.15, 0.68, 0.79, 0.11, 1.0),  # diffusion
    State(0.10, 0.55, 0.65, 0.16, 0.0),  # halt condition
    State(0.05, 0.81, 0.90, 0.07, 1.0),  # recovered
]

regulator = ShadowRegulator()
receipts = [regulator.decide(s) for s in trajectory]
pd.DataFrame([{**r['state'], **{k:v for k,v in r.items() if k != 'state'}} for r in receipts])


## Shadow A/B protocol
For each task, run the same seeds in two arms:

1. Baseline orchestration with no regulator
2. Shadow regulator that records the action it would recommend

Do not apply the recommendations in the first validation round. Compare recommendations with observed outcomes. Only after calibration should a second experiment apply novelty pulses and re-anchoring.

In [ ]:
def classify_outcome(s: State):
    if s.purpose_integrity < 1.0 or s.coherence < 0.70:
        return 'diffusion'
    if s.diversity < 0.04 and s.epistemic_gain <= 0:
        return 'stasis_or_false_stability'
    if s.returnability >= 0.82 and s.epistemic_gain > 0:
        return 'productive_exploration'
    return 'indeterminate'

for r, s in zip(receipts, trajectory):
    r['observed_outcome'] = classify_outcome(s)

with open('/content/shadow_regulator_receipts.jsonl', 'w', encoding='utf-8') as f:
    for r in receipts:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('Saved shadow receipts')
